# Multiple Linear Regression + Feature Scaling

**Predicting student final grade `G3` from several features at once.**

This is the multi-feature sibling of `student-result-prediction.ipynb`.

In the single-feature notebook the model was:

$$f_{w,b}(x) = w\,x + b$$

Now we have `n` features, so `w` becomes a **vector** and we use a dot product (this is exactly Andrew Ng Course 1, Week 2):

$$f_{\vec{w},b}(\vec{x}) = \vec{w}\cdot\vec{x} + b = w_1 x_1 + w_2 x_2 + \dots + w_n x_n + b$$

Two new ideas vs. the single-feature notebook:
1. **Vectorization** — using `np.dot` / matrix math instead of looping over features.
2. **Feature scaling (z-score normalization)** — because features live on very different ranges (e.g. `age` 15-22 vs `studytime` 1-4), gradient descent converges much faster when every feature is on a comparable scale.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

In [ ]:
# Same dataset as the single-feature notebook
# df = pd.read_csv("./student-por.csv")
df = pd.read_csv("https://raw.githubusercontent.com/naimdotcom/ML-everything/main/Student-score-result/student-por.csv")
df.head()

## 1. Pick multiple features

Instead of just `absences`, we feed the model several numeric columns. These are all plausibly related to the final grade `G3`:

| feature | meaning |
|---|---|
| `G1` | first period grade |
| `G2` | second period grade |
| `studytime` | weekly study time (1-4) |
| `failures` | number of past class failures |
| `absences` | number of school absences |
| `age` | student's age |
| `Medu` | mother's education (0-4) |
| `Fedu` | father's education (0-4) |

Notice the ranges are wildly different — that is exactly why we need feature scaling.

In [ ]:
feature_cols = ["G1", "G2", "studytime", "failures", "absences", "age", "Medu", "Fedu"]
target_col = "G3"

# Some columns in this CSV are read as strings (quoted numbers), so force numeric
X = df[feature_cols].apply(pd.to_numeric, errors="coerce").values  # shape (m, n)
y = df[target_col].apply(pd.to_numeric, errors="coerce").values    # shape (m,)

print("X shape:", X.shape, "  y shape:", y.shape)
print("Number of features (n):", X.shape[1])

In [ ]:
# See how different the feature ranges are -> motivation for scaling
pd.DataFrame(X, columns=feature_cols).describe().loc[["min", "max", "mean", "std"]]

## 1b. Explore the data visually

Before modelling, it helps to *look* at the data. The plots below answer three questions:
- How is each feature distributed? (histograms)
- Which features are most correlated with the target `G3`? (heatmap)
- What does the relationship look like in 3D for the two strongest predictors? (3D scatter)

In [ ]:
# Histogram of every feature + the target, so we can see distributions and ranges
plot_df = pd.DataFrame(np.column_stack([X, y]), columns=feature_cols + [target_col])

fig, axes = plt.subplots(3, 3, figsize=(13, 10))
for ax, col in zip(axes.ravel(), plot_df.columns):
    ax.hist(plot_df[col], bins=20, color="steelblue", edgecolor="white")
    ax.set_title(col)
# hide any unused subplot
for ax in axes.ravel()[len(plot_df.columns):]:
    ax.axis("off")
fig.suptitle("Distribution of each feature and the target G3", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap: how strongly each pair of variables moves together.
# The bottom row / last column (G3) tells us which features predict the grade best.
corr = plot_df.corr()

fig, ax = plt.subplots(figsize=(9, 7))
im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=45, ha="right")
ax.set_yticklabels(corr.columns)

# annotate each cell with the correlation value
for i in range(len(corr.columns)):
    for j in range(len(corr.columns)):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center",
                color="white" if abs(corr.iloc[i, j]) > 0.5 else "black", fontsize=8)

fig.colorbar(im, ax=ax, label="correlation")
ax.set_title("Correlation heatmap (look at the G3 row/column)")
plt.tight_layout()
plt.show()

print("Correlation of each feature with G3 (sorted):")
print(corr[target_col].drop(target_col).sort_values(ascending=False).round(3))

In [ ]:
# 3D scatter: the two strongest predictors (G1, G2) against the target G3.
# With one feature we plotted a 2D scatter; with two features the data lives in 3D.
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401  (enables 3d projection)

fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection="3d")

g1 = plot_df["G1"].values
g2 = plot_df["G2"].values
p = ax.scatter(g1, g2, y, c=y, cmap="viridis", alpha=0.6)

ax.set_xlabel("G1 (period 1 grade)")
ax.set_ylabel("G2 (period 2 grade)")
ax.set_zlabel("G3 (final grade)")
ax.set_title("3D view: G1 & G2 vs final grade G3")
fig.colorbar(p, ax=ax, shrink=0.6, label="G3")
ax.view_init(elev=20, azim=-60)   # change these angles to rotate the view
plt.show()

## 2. Train / test split

In [ ]:
X_train, X_test, Y_train, Y_test = train_test_split(X, y, test_size=0.33, random_state=42)
print(X_train.shape, Y_train.shape, X_test.shape, Y_test.shape)

## 3. Feature scaling (z-score normalization)

For each feature `j` we compute:

$$x^{(i)}_j \;\rightarrow\; \frac{x^{(i)}_j - \mu_j}{\sigma_j}$$

where $\mu_j$ is the mean and $\sigma_j$ is the standard deviation **of that feature**.

**Important rule (no data leakage):** compute $\mu$ and $\sigma$ on the *training* set only, then reuse those same numbers to scale the test set. The test set must be treated as "unseen" data.

`axis=0` means "compute the statistic down each column", i.e. one mean/std per feature.

In [ ]:
def zscore_normalize(X, mu, sigma):
    return (X - mu) / sigma

# Stats from TRAINING data only
mu = X_train.mean(axis=0)      # shape (n,)
sigma = X_train.std(axis=0)    # shape (n,)

X_train_norm = zscore_normalize(X_train, mu, sigma)
X_test_norm = zscore_normalize(X_test, mu, sigma)   # reuse training mu/sigma!

print("Per-feature mean (mu):  ", np.round(mu, 2))
print("Per-feature std (sigma):", np.round(sigma, 2))
print("\nAfter scaling, each feature has ~0 mean and ~1 std:")
print("train mean:", np.round(X_train_norm.mean(axis=0), 3))
print("train std: ", np.round(X_train_norm.std(axis=0), 3))

In [ ]:
# Visualize the effect of scaling on two features with very different ranges
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].scatter(X_train[:, 0], X_train[:, 5], alpha=0.4)  # G1 vs age
axes[0].set_title("Before scaling")
axes[0].set_xlabel("G1")
axes[0].set_ylabel("age")

axes[1].scatter(X_train_norm[:, 0], X_train_norm[:, 5], alpha=0.4)
axes[1].set_title("After z-score scaling")
axes[1].set_xlabel("G1 (scaled)")
axes[1].set_ylabel("age (scaled)")

plt.tight_layout()
plt.show()

## 4. Cost function (vectorized)

Same squared-error cost as before, but now the prediction is a dot product over all features.

$$J(\vec{w}, b) = \frac{1}{2m} \sum_{i=1}^{m} \left( f_{\vec{w},b}(\vec{x}^{(i)}) - y^{(i)} \right)^2$$

`X @ w` computes all `m` predictions at once — that's the vectorized form of the loop you'd otherwise write.

In [ ]:
def compute_cost(X, y, w, b):
    m = X.shape[0]
    f_wb = X @ w + b            # (m,) predictions for every example
    cost = np.sum((f_wb - y) ** 2) / (2 * m)
    return cost

## 5. Gradient (vectorized)

For multiple features the gradients are:

$$\frac{\partial J}{\partial w_j} = \frac{1}{m} \sum_{i=1}^{m} \left( f_{\vec{w},b}(\vec{x}^{(i)}) - y^{(i)} \right) x^{(i)}_j \qquad \frac{\partial J}{\partial b} = \frac{1}{m} \sum_{i=1}^{m} \left( f_{\vec{w},b}(\vec{x}^{(i)}) - y^{(i)} \right)$$

The `w` gradient is now a **vector** of length `n` (one partial derivative per feature). `X.T @ err` does the whole sum-over-examples for every feature in one matrix multiply.

In [ ]:
def compute_gradient(X, y, w, b):
    m = X.shape[0]
    err = (X @ w + b) - y       # (m,)
    dj_dw = (X.T @ err) / m     # (n,)  one gradient per feature
    dj_db = np.sum(err) / m     # scalar
    return dj_dw, dj_db

## 6. Gradient descent

Identical loop to the single-feature notebook — the only change is that `w` starts as a zero **vector** of length `n` instead of a single scalar.

In [ ]:
def gradient_descent(X, y, alpha=0.01, iterations=1000):
    n = X.shape[1]
    w = np.zeros(n)            # vector now, not a scalar
    b = 0.0
    cost_history = []
    for i in range(iterations):
        dj_dw, dj_db = compute_gradient(X, y, w, b)
        w = w - alpha * dj_dw
        b = b - alpha * dj_db
        cost_history.append(compute_cost(X, y, w, b))
    return w, b, cost_history

In [ ]:
# Because features are scaled, we can use a much larger learning rate than 0.001
W, B, cost_history = gradient_descent(X_train_norm, Y_train, alpha=0.1, iterations=2000)

print("Learned weights (one per feature):")
for name, weight in zip(feature_cols, W):
    print(f"  {name:>10}: {weight:+.4f}")
print(f"\nBias b: {B:.4f}")
print(f"\nFirst cost: {cost_history[0]:.4f}   Final cost: {cost_history[-1]:.4f}")

Because the features are scaled, the **magnitude of each weight is now comparable** — a bigger absolute weight roughly means that feature matters more for the prediction. (`G2` and `G1` should dominate, which makes sense: the earlier grades strongly predict the final grade.)

## 7. Learning curve

Cost should drop quickly and then flatten. If it ever goes *up* or oscillates, the learning rate is too large.

In [ ]:
plt.plot(cost_history)
plt.axhline(y=cost_history[-1], color="red", linestyle="--",
            label=f"Final cost: {cost_history[-1]:.4f}")
plt.xlabel("iteration")
plt.ylabel("cost J")
plt.title("Learning curve (scaled features)")
plt.legend()
plt.show()

## 8. Evaluate on train vs test

In [ ]:
train_cost = compute_cost(X_train_norm, Y_train, W, B)
test_cost = compute_cost(X_test_norm, Y_test, W, B)
print(f"Train cost: {train_cost:.4f}")
print(f"Test cost:  {test_cost:.4f}")

def r2_score(X, y, w, b):
    pred = X @ w + b
    ss_res = np.sum((y - pred) ** 2)
    ss_tot = np.sum((y - y.mean()) ** 2)
    return 1 - ss_res / ss_tot

print(f"\nTrain R^2: {r2_score(X_train_norm, Y_train, W, B):.4f}")
print(f"Test  R^2: {r2_score(X_test_norm, Y_test, W, B):.4f}")

In [ ]:
# Predicted vs actual on the test set. Perfect predictions lie on the red line.
test_pred = X_test_norm @ W + B
plt.scatter(Y_test, test_pred, alpha=0.4)
lims = [Y_test.min(), Y_test.max()]
plt.plot(lims, lims, color="red", label="perfect prediction")
plt.xlabel("actual G3")
plt.ylabel("predicted G3")
plt.title("Predicted vs Actual (test set)")
plt.legend()
plt.show()

## 8b. More diagnostic graphs

In the single-feature notebook you fit a **line**. With many features the model is a **hyperplane** we can't fully draw. But we can still visualize:
- which features the model leans on (weight bar chart),
- whether the errors are healthy (residual plots),
- and the fitted **plane** if we train a small 2-feature model on the top predictors.

In [ ]:
# Bar chart of the learned weights. Because features are scaled, the bar HEIGHT
# is directly comparable -> a taller bar = more influence on the prediction.
order = np.argsort(np.abs(W))[::-1]
sorted_names = [feature_cols[i] for i in order]
sorted_w = W[order]
colors = ["seagreen" if v >= 0 else "indianred" for v in sorted_w]

plt.figure(figsize=(9, 5))
plt.bar(sorted_names, sorted_w, color=colors)
plt.axhline(0, color="black", linewidth=0.8)
plt.ylabel("weight value")
plt.title("Feature importance (scaled weights)\ngreen = pushes grade up, red = pushes grade down")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
# Residuals = actual - predicted. For a good model they should be:
#   (left)  scattered randomly around 0 with no pattern vs the prediction
#   (right) roughly bell-shaped (normally distributed) around 0
train_pred = X_train_norm @ W + B
residuals = Y_train - train_pred

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

axes[0].scatter(train_pred, residuals, alpha=0.4)
axes[0].axhline(0, color="red")
axes[0].set_xlabel("predicted G3")
axes[0].set_ylabel("residual (actual - predicted)")
axes[0].set_title("Residuals vs predictions")

axes[1].hist(residuals, bins=30, color="steelblue", edgecolor="white")
axes[1].axvline(0, color="red")
axes[1].set_xlabel("residual")
axes[1].set_ylabel("count")
axes[1].set_title("Distribution of residuals")

plt.tight_layout()
plt.show()

### The 3D regression *plane*

With **one** feature, the model is a line `w*x + b`.
With **two** features, the model becomes a flat **plane**: `w1*x1 + w2*x2 + b`.
(With 8 features it's an 8D hyperplane we can't draw — so we train a small 2-feature model on `G1` and `G2` purely to *see* the geometry.)

The red surface below is the fitted plane; the dots are the real students. Their vertical distance to the plane is the residual.

In [ ]:
# Train a small 2-feature model on G1 and G2 so we can draw the fitted plane.
two = ["G1", "G2"]
idx = [feature_cols.index(c) for c in two]

X2_train = X_train[:, idx]
mu2 = X2_train.mean(axis=0)
sigma2 = X2_train.std(axis=0)
X2_train_norm = zscore_normalize(X2_train, mu2, sigma2)

W2, B2, _ = gradient_descent(X2_train_norm, Y_train, alpha=0.1, iterations=2000)
print("2-feature model -> weights:", np.round(W2, 3), " bias:", round(B2, 3))

# Build a grid over the G1/G2 range and predict G3 on it -> that's the plane
g1_line = np.linspace(X2_train[:, 0].min(), X2_train[:, 0].max(), 30)
g2_line = np.linspace(X2_train[:, 1].min(), X2_train[:, 1].max(), 30)
G1g, G2g = np.meshgrid(g1_line, g2_line)

grid = np.column_stack([G1g.ravel(), G2g.ravel()])
grid_norm = zscore_normalize(grid, mu2, sigma2)
G3g = (grid_norm @ W2 + B2).reshape(G1g.shape)

fig = plt.figure(figsize=(11, 8))
ax = fig.add_subplot(111, projection="3d")

# actual training points
ax.scatter(X2_train[:, 0], X2_train[:, 1], Y_train, c="navy", alpha=0.35, label="actual students")
# fitted plane
ax.plot_surface(G1g, G2g, G3g, color="red", alpha=0.4)

ax.set_xlabel("G1")
ax.set_ylabel("G2")
ax.set_zlabel("G3 (final grade)")
ax.set_title("Fitted regression PLANE over G1 & G2")
ax.view_init(elev=18, azim=-60)
plt.show()

## 9. Sanity check against scikit-learn

Our from-scratch weights should closely match `LinearRegression`. Small differences are just because gradient descent stopped at a finite number of iterations while sklearn solves it exactly.

In [ ]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X_train_norm, Y_train)

print("Comparison of weights:")
print(f"{'feature':>10} | {'ours':>10} | {'sklearn':>10}")
print("-" * 36)
for name, ours, skl in zip(feature_cols, W, model.coef_):
    print(f"{name:>10} | {ours:>+10.4f} | {skl:>+10.4f}")
print(f"{'bias':>10} | {B:>+10.4f} | {model.intercept_:>+10.4f}")

print(f"\nsklearn test R^2: {model.score(X_test_norm, Y_test):.4f}")

## 10. Predict on a new student

To predict for a brand-new student, scale their raw feature values with the **same training `mu` and `sigma`**, then apply the model.

In [ ]:
# Order must match feature_cols:
# [G1, G2, studytime, failures, absences, age, Medu, Fedu]
new_student = np.array([12, 13, 2, 0, 4, 17, 3, 2], dtype=float)

new_student_norm = zscore_normalize(new_student, mu, sigma)
predicted_g3 = new_student_norm @ W + B

print("Raw features:   ", dict(zip(feature_cols, new_student)))
print(f"\nPredicted final grade G3: {predicted_g3:.2f}")

## Recap — what changed vs single-feature regression

| | Single-feature | Multi-feature (this notebook) |
|---|---|---|
| input `x` | one column | matrix `X` of shape `(m, n)` |
| weight `w` | scalar | vector of length `n` |
| prediction | `w*x + b` | `X @ w + b` (dot product) |
| scaling | scaled one column | **z-score per feature** (`axis=0`) |
| learning rate | 0.001 | 0.1 (faster, thanks to scaling) |

**Key takeaways**
1. Vectorization lets the *same* cost/gradient code handle any number of features.
2. Feature scaling makes gradient descent converge far faster and makes the weights comparable.
3. Always fit the scaler (`mu`, `sigma`) on the training set and reuse it on test / new data.